In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [2]:
# ============================================
# NOTEBOOK: WALK-FORWARD TRIADA (LGB + XGB + CAT)
# ============================================
# Cel: Walk-forward ensemble trzech modeli z multi-seed
# ============================================

# ============================================
# SEKCJA 1: IMPORTS + CONFIG
# ============================================
import numpy as np
import polars as pl
from pathlib import Path
import os
import time
import datetime
import warnings
warnings.filterwarnings('ignore')

import lightgbm as lgb
import xgboost as xgb
from catboost import CatBoostRegressor

pl.Config.set_tbl_rows(100)
pl.Config.set_tbl_cols(100)

# ============================================
# KONFIGURACJA
# ============================================

# Okna walk-forward
WINDOW_SIZE = 500
STEP = 100

# Zakres danych
MIN_TS = 1
MAX_TS_TRAIN = 3601

# Multi-seed (3 seedy)
SEEDS = [42, 777, 1337]

# Horyzonty
HORIZONS = [1, 3, 10, 25]

# LightGBM (sprawdzona konfiguracja)
LGB_PARAMS = {
    'objective': 'regression',
    'metric': 'rmse',
    'num_leaves': 50,
    'learning_rate': 0.05,
    'n_estimators': 300,
    'max_depth': 10,
    'min_child_samples': 20,
    'subsample': 0.8,
    'colsample_bytree': 0.8,
    'reg_alpha': 0.1,
    'reg_lambda': 0.1,
    'verbose': -1
}

# XGBoost
XGB_PARAMS = {
    'objective': 'reg:squarederror',
    'eval_metric': 'rmse',
    'n_estimators': 300,
    'learning_rate': 0.05,
    'max_depth': 10,
    'min_child_weight': 20,
    'subsample': 0.8,
    'colsample_bytree': 0.8,
    'reg_alpha': 0.1,
    'reg_lambda': 0.1,
    'verbosity': 0
}

# CatBoost
CAT_PARAMS = {
    'iterations': 300,
    'learning_rate': 0.05,
    'depth': 8,
    'l2_leaf_reg': 1.0,
    'verbose': False
}

print("="*60)
print("KONFIGURACJA WALK-FORWARD TRIADA")
print("="*60)
print(f"Window size: {WINDOW_SIZE}, Step: {STEP}")
print(f"Seeds: {SEEDS}")
print(f"Horyzonty: {HORIZONS}")
print("="*60)

# ============================================
# SEKCJA 2: WCZYTANIE DANYCH
# ============================================
print("\n" + "="*60)
print("WCZYTANIE DANYCH")
print("="*60)

KAGGLE_ENV = os.path.exists('/kaggle/input')

if KAGGLE_ENV:
    train_path = '/kaggle/input/competitions/ts-forecasting/train.parquet'
    test_path = '/kaggle/input/competitions/ts-forecasting/test.parquet'
else:
    base_dir = Path("..")
    train_path = base_dir / "data" / "train.parquet"
    test_path = base_dir / "data" / "test.parquet"

train_full = pl.read_parquet(train_path)
test_full = pl.read_parquet(test_path)

print(f"✅ Train: {train_full.shape}")
print(f"✅ Test: {test_full.shape}")

# ============================================
# SEKCJA 3: TYPE CONVERSION + BRUDNE/CZYSTE
# ============================================
print("\n🔄 Konwersja typów...")

categorical_cols = ['code', 'sub_code', 'sub_category']
for col in categorical_cols:
    if col in train_full.columns:
        train_full = train_full.with_columns(pl.col(col).cast(pl.Categorical))
    if col in test_full.columns:
        test_full = test_full.with_columns(pl.col(col).cast(pl.Categorical))

int_cols = ['horizon', 'ts_index']
for col in int_cols:
    if col in train_full.columns:
        train_full = train_full.with_columns(pl.col(col).cast(pl.Int16))
    if col in test_full.columns:
        test_full = test_full.with_columns(pl.col(col).cast(pl.Int16))

feature_cols = [c for c in train_full.columns if c.startswith('feature_')]
for col in feature_cols:
    if col in train_full.columns and train_full[col].dtype == pl.Float64:
        train_full = train_full.with_columns(pl.col(col).cast(pl.Float32))
    if col in test_full.columns and test_full[col].dtype == pl.Float64:
        test_full = test_full.with_columns(pl.col(col).cast(pl.Float32))

print(f"✅ Typy skonwertowane, cechy: {len(feature_cols)}")

# Stwórz kopię czystą (dla XGBoost i CatBoost)
train_czysty = train_full.clone()
test_czysty = test_full.clone()

# Wypełnij NaN dla czystej kopii (1e-8)
for c in feature_cols:
    if train_czysty[c].null_count() > 0:
        train_czysty = train_czysty.with_columns(pl.col(c).fill_null(1e-8).alias(c))
    if c in test_czysty.columns and test_czysty[c].null_count() > 0:
        test_czysty = test_czysty.with_columns(pl.col(c).fill_null(1e-8).alias(c))

print("✅ Brudne dane (z NaN) dla LightGBM")
print("✅ Czyste dane (1e-8) dla XGBoost i CatBoost")

# ============================================
# SEKCJA 4: METRYKA
# ============================================
def _clip01(x: float) -> float:
    return float(np.minimum(np.maximum(x, 0.0), 1.0))

def weighted_rmse_score(y_target, y_pred, w) -> float:
    denom = np.sum(w * y_target ** 2)
    if denom == 0:
        return 0.0
    ratio = np.sum(w * (y_target - y_pred) ** 2) / denom
    clipped = _clip01(ratio)
    val = 1.0 - clipped
    return float(np.sqrt(val))

print("✅ Metryka zdefiniowana")

# ============================================
# SEKCJA 5: GENEROWANIE OKIEN
# ============================================
windows = []
start = MIN_TS
while start + WINDOW_SIZE <= MAX_TS_TRAIN:
    train_end = start + WINDOW_SIZE
    windows.append({
        'train_start': start,
        'train_end': train_end,
    })
    start += STEP

print(f"\n📊 Liczba okien: {len(windows)}")

# ============================================
# SEKCJA 6: WALK-FORWARD TRENING (3 MODELE × 3 SEEDY)
# ============================================
print("\n" + "="*60)
print("WALK-FORWARD TRENING")
print("="*60)

# Storage dla predykcji
lgb_predictions = []   # lista (horizon, seed, window_idx, preds)
xgb_predictions = []
cat_predictions = []

total_combinations = len(HORIZONS) * len(SEEDS) * len(windows)
counter = 0

for w_idx, window in enumerate(windows):
    print(f"\n{'='*50}")
    print(f"Okno {w_idx+1}/{len(windows)}: train {window['train_start']}-{window['train_end']}")
    print(f"{'='*50}")
    
    for horizon in HORIZONS:
        for seed in SEEDS:
            counter += 1
            print(f"  [{counter}/{total_combinations}] H={horizon}, seed={seed}...", end=" ")
            
            # ========== LIGHTGBM (brudne dane) ==========
            train_h = train_full.filter(
                (pl.col('horizon') == horizon) &
                (pl.col('ts_index') >= window['train_start']) &
                (pl.col('ts_index') <= window['train_end'])
            ).sort('ts_index')
            
            if len(train_h) >= 100:
                params = LGB_PARAMS.copy()
                params['random_state'] = seed
                model = lgb.LGBMRegressor(**params)
                
                X = train_h.select(feature_cols).to_numpy()
                y = train_h.select('y_target').to_numpy().ravel()
                w = train_h.select('weight').to_numpy().ravel()
                
                model.fit(X, y, sample_weight=w)
                
                # Predykcja na test
                test_h = test_full.filter(pl.col('horizon') == horizon)
                X_test = test_h.select(feature_cols).to_numpy()
                pred = model.predict(X_test)
                lgb_predictions.append((horizon, seed, w_idx, pred))
                print("LGB ✓", end=" ")
            
            # ========== XGBOOST (czyste dane) ==========
            train_h_clean = train_czysty.filter(
                (pl.col('horizon') == horizon) &
                (pl.col('ts_index') >= window['train_start']) &
                (pl.col('ts_index') <= window['train_end'])
            ).sort('ts_index')
            
            if len(train_h_clean) >= 100:
                params = XGB_PARAMS.copy()
                params['random_state'] = seed
                model = xgb.XGBRegressor(**params)
                
                X = train_h_clean.select(feature_cols).to_numpy()
                y = train_h_clean.select('y_target').to_numpy().ravel()
                w = train_h_clean.select('weight').to_numpy().ravel()
                
                model.fit(X, y, sample_weight=w, verbose=False)
                
                test_h = test_czysty.filter(pl.col('horizon') == horizon)
                X_test = test_h.select(feature_cols).to_numpy()
                pred = model.predict(X_test)
                xgb_predictions.append((horizon, seed, w_idx, pred))
                print("XGB ✓", end=" ")
            
            # ========== CATBOOST (czyste dane) ==========
            if len(train_h_clean) >= 100:
                params = CAT_PARAMS.copy()
                params['random_seed'] = seed
                model = CatBoostRegressor(**params)
                
                X = train_h_clean.select(feature_cols + categorical_cols).to_pandas()
                y = train_h_clean.select('y_target').to_numpy().ravel()
                w = train_h_clean.select('weight').to_numpy().ravel()
                
                model.fit(X, y, sample_weight=w, cat_features=categorical_cols, verbose=False)
                
                test_h = test_czysty.filter(pl.col('horizon') == horizon)
                X_test = test_h.select(feature_cols + categorical_cols).to_pandas()
                pred = model.predict(X_test)
                cat_predictions.append((horizon, seed, w_idx, pred))
                print("CAT ✓", end=" ")
            
            print()

print(f"\n✅ Wytrenowane:")
print(f"   LightGBM: {len(lgb_predictions)} modeli")
print(f"   XGBoost: {len(xgb_predictions)} modeli")
print(f"   CatBoost: {len(cat_predictions)} modeli")

# ============================================
# SEKCJA 7: ENSEMBLE (średnia po wszystkich modelach)
# ============================================
print("\n" + "="*60)
print("ENSEMBLE PREDYKCJI")
print("="*60)

final_preds = {}

for horizon in HORIZONS:
    # Zbierz wszystkie predykcje dla tego horizonu
    all_preds = []
    
    for (h, seed, w_idx, pred) in lgb_predictions:
        if h == horizon:
            all_preds.append(pred)
    
    for (h, seed, w_idx, pred) in xgb_predictions:
        if h == horizon:
            all_preds.append(pred)
    
    for (h, seed, w_idx, pred) in cat_predictions:
        if h == horizon:
            all_preds.append(pred)
    
    if all_preds:
        final_pred = np.mean(all_preds, axis=0)
        final_preds[horizon] = final_pred
        print(f"Horyzont {horizon}: {len(all_preds)} modeli → ensemble")

# ============================================
# SEKCJA 8: SUBMISSION
# ============================================
print("\n" + "="*60)
print("SUBMISSION")
print("="*60)

all_ids = []
all_preds = []

for horizon in HORIZONS:
    test_h = test_full.filter(pl.col('horizon') == horizon)
    ids = test_h.select('id').to_numpy().ravel()
    
    if horizon in final_preds:
        preds = final_preds[horizon]
        all_ids.extend(ids)
        all_preds.extend(preds)
    else:
        print(f"⚠️ Brak predykcji dla horizon {horizon}")

submission_df = pl.DataFrame({
    'id': all_ids,
    'prediction': all_preds
})

timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
submission_path = f"/kaggle/working/walkforward_triad_{timestamp}.csv"
submission_df.write_csv(submission_path)

print(f"✅ Submission saved: {submission_path}")
print(f"📊 Shape: {submission_df.shape}")
print(f"📊 Prediction range: [{min(all_preds):.4f}, {max(all_preds):.4f}]")
print(submission_df.head())

# ============================================
# PODSUMOWANIE
# ============================================
print("\n" + "="*80)
print("WALK-FORWARD TRIADA - PODSUMOWANIE")
print("="*80)
print(f"✅ Okna: {len(windows)}")
print(f"✅ LightGBM: {len(lgb_predictions)} modeli")
print(f"✅ XGBoost: {len(xgb_predictions)} modeli")
print(f"✅ CatBoost: {len(cat_predictions)} modeli")
print(f"✅ Ensemble: {sum(len(final_preds) for _ in final_preds)} horyzontów")
print(f"✅ Submission: {submission_path}")
print("="*80)

KONFIGURACJA WALK-FORWARD TRIADA
Window size: 500, Step: 100
Seeds: [42, 777, 1337]
Horyzonty: [1, 3, 10, 25]

WCZYTANIE DANYCH
✅ Train: (5337414, 94)
✅ Test: (1447107, 92)

🔄 Konwersja typów...
✅ Typy skonwertowane, cechy: 86
✅ Brudne dane (z NaN) dla LightGBM
✅ Czyste dane (1e-8) dla XGBoost i CatBoost
✅ Metryka zdefiniowana

📊 Liczba okien: 32

WALK-FORWARD TRENING

Okno 1/32: train 1-501
  [1/384] H=1, seed=42... LGB ✓ XGB ✓ CAT ✓ 
  [2/384] H=1, seed=777... LGB ✓ XGB ✓ CAT ✓ 
  [3/384] H=1, seed=1337... LGB ✓ XGB ✓ CAT ✓ 
  [4/384] H=3, seed=42... LGB ✓ XGB ✓ CAT ✓ 
  [5/384] H=3, seed=777... LGB ✓ XGB ✓ CAT ✓ 
  [6/384] H=3, seed=1337... LGB ✓ XGB ✓ CAT ✓ 
  [7/384] H=10, seed=42... LGB ✓ XGB ✓ CAT ✓ 
  [8/384] H=10, seed=777... LGB ✓ XGB ✓ CAT ✓ 
  [9/384] H=10, seed=1337... LGB ✓ XGB ✓ CAT ✓ 
  [10/384] H=25, seed=42... LGB ✓ XGB ✓ CAT ✓ 
  [11/384] H=25, seed=777... LGB ✓ XGB ✓ CAT ✓ 
  [12/384] H=25, seed=1337... LGB ✓ XGB ✓ CAT ✓ 

Okno 2/32: train 101-601
  [13/384] H=1, se